# Isolated V2 Mem adapter: Prepare Data & Train on Kaggle (With Auto-Resume & 2-Dataset Support)

Notebook này được tối ưu cho **Kaggle 2x T4 GPUs** với các cải tiến quan trọng:
1. **Tương tác 2 Datasets**: Tự động liên kết 10 base LIBERO-10 demonstrations từ Dataset 1 (đã tải lên trước đó) và derived mem demonstrations từ Dataset 2 (`libero_mem_derived_v1_sources.zip`), không cần upload lại dữ liệu base nặng hàng chục GB.
2. **Xử lý Perception Song Song (Parallel 2x GPU)**: Chạy song song T1 trên GPU 0 và T3 trên GPU 1, giảm thời gian chuẩn bị dữ liệu từ ~10-12h xuống còn ~4.5-5h.
3. **Quản lý Phiên 12h Kaggle (Session Safety)**: Thiết kế modular qua cờ `RUN_DATA_PREP` và `RUN_TRAINING`. Cho phép chia 2 sessions an toàn (Session 1: Data Prep ~4.5h -> Save; Session 2: Train 5,000 steps ~8h) hoặc chạy liền mạch.
4. **Tăng Steps Huấn Luyện**: Mặc định 5,000 steps (dễ dàng cấu hình lên 6,000+) để SmolVLA hội tụ sâu hơn.
5. **Cơ chế Auto-Resume Toàn Diện**: Tự động phục hồi cả tiến trình perception (theo từng episode) lẫn training loop (tự nhận diện `checkpoint-XXXXXX` cao nhất để tiếp tục).

### Hướng dẫn thiết lập trên Kaggle:
1. **GPU**: Chọn **2x T4 GPUs** (Accelerator: GPU T4 x2).
2. **Internet**: Bật **Internet On**.
3. **Inputs cần gắn vào Notebook**:
   - **Dataset 1**: Dataset chứa 10 base demonstrations LIBERO-10 (đã upload trước đó).
   - **Dataset 2**: Dataset chứa derived mem (`libero_mem_derived_v1_sources.zip` hoặc folder).
   - **Mã nguồn**: Dataset hoặc file ZIP chứa code (`mem_adapter_v1_code.zip` hoặc `memory_sft_code`).


In [1]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile, tempfile

# ==============================================================================
# ⚙️ CẤU HÌNH THI CÔNG & QUẢN LÝ PHIÊN KAGGLE (12-HOUR SESSION MANAGEMENT)
# ==============================================================================
# Chiến lược chạy 2 sessions an toàn hoặc 1 session liền mạch:
# - Session 1 (Data Prep): RUN_DATA_PREP = True,  RUN_TRAINING = False (~4.5h)
#   Output lưu shards tại /kaggle/working/mem-adapter-v1/dataset. Bấm Save Version.
# - Session 2 (Training):  RUN_DATA_PREP = False, RUN_TRAINING = True  (~8h cho 5,000 steps)
#   Gắn output Session 1 làm Input, notebook sẽ bỏ qua prep và train ngay!
# - Chạy liền mạch:       RUN_DATA_PREP = True,  RUN_TRAINING = True
RUN_DATA_PREP = True          # Tạo perception memory & đóng gói training shards
RUN_TRAINING = True           # Huấn luyện SmolVLA memory adapter
TRAIN_STEPS = 5000            # Số optimizer steps (tăng từ 3000 -> 5000 để hội tụ sâu hơn)
SAVE_EVERY = 500              # Tần suất lưu checkpoint (mỗi 500 steps)
AUTO_RESUME_TRAINING = True   # Tự động tìm checkpoint cao nhất để resume nếu phiên bị ngắt
GPUS = 2

INPUT = Path('/kaggle/input')
SCRATCH = Path('/kaggle/temp/mem-adapter-v1')
WORK = Path('/kaggle/working/mem-adapter-v1')
PREP = WORK / 'preparation'
DATA = WORK / 'dataset'
BASE_ROOT = SCRATCH / 'base_demonstrations'
ASSETS = SCRATCH / 'assets'
REPO = SCRATCH / 'code'

EXPECTED_LOCK = '3d9be02b8861481f0b4379697649198c01c36aec5915f6f50b9e53c23e0da2cf'
PROCESSING_RESERVE_GIB = 50

for d in (SCRATCH, WORK, PREP, BASE_ROOT, ASSETS, REPO):
    d.mkdir(parents=True, exist_ok=True)

def run(python, *args):
    subprocess.run([str(python), *map(str, args)], cwd=REPO, check=True)

def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def extract_checked(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target_path = (destination / member.filename).resolve()
            if not target_path.is_relative_to(destination.resolve()):
                raise ValueError(f'Archive path escapes destination: {member.filename}')
        z.extractall(destination)

# Khởi tạo mã nguồn vào REPO
if not (REPO / 'scripts/train_mem_adapter.py').is_file():
    code_zips = sorted(INPUT.rglob('*code*.zip')) + sorted(INPUT.rglob('mem_adapter*.zip'))
    found = False
    for cz in code_zips:
        try:
            with zipfile.ZipFile(cz) as z:
                if any('train_mem_adapter.py' in n for n in z.namelist()):
                    print(f'Extracting code bundle: {cz} -> {REPO}...')
                    extract_checked(cz, REPO)
                    found = True
                    break
        except Exception:
            continue
    if not found:
        candidates = [p.parents[1] for p in INPUT.rglob('scripts/train_mem_adapter.py')]
        for c in candidates:
            if (c / 'scripts/train_mem_adapter.py').is_file():
                print(f'Copying code from {c} -> {REPO}...')
                shutil.copytree(c, REPO, dirs_exist_ok=True)
                found = True
                break
    assert (REPO / 'scripts/train_mem_adapter.py').is_file(), 'Không tìm thấy source code trong /kaggle/input!'
print('✓ Source code đã sẵn sàng tại:', REPO)


Copying code from /kaggle/input/datasets/luquanglinh/mem-adapter-v1/mem_adapter_v1_code -> /kaggle/temp/mem-adapter-v1...
✓ Source code đã sẵn sàng tại: /kaggle/temp/mem-adapter-v1


In [2]:
import subprocess, sys, shutil, os
from pathlib import Path

ENV_DIR = SCRATCH / 'perception-env'
PERCEPTION_PY = ENV_DIR / 'bin/python'
TRAIN_PY = sys.executable
PIP_BIN = ENV_DIR / 'bin/pip'

# Khởi tạo perception-env chuẩn bằng virtualenv (độc lập, chứa Transformers 4.51.3)
if not PERCEPTION_PY.is_file():
    print('1. Đang tạo perception-env bằng virtualenv...')
    shutil.rmtree(ENV_DIR, ignore_errors=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv'], check=True)
    subprocess.run([sys.executable, '-m', 'virtualenv', str(ENV_DIR)], check=True)
    subprocess.run([str(PIP_BIN), 'install', '-q', 'hf_xet'], check=True)

    print('2. Đang cài đặt thư viện vào perception-env...')
    clean_env = dict(os.environ, PYTHONNOUSERSITE='1')
    req_path = REPO / 'envs/memory_perception.txt'
    subprocess.run([
        str(PIP_BIN), 'install', '--no-cache-dir', '-r', str(req_path)
    ], check=True, env=clean_env)

print('3. Kiểm tra xác nhận perception-env...')
clean_env = dict(os.environ, PYTHONNOUSERSITE='1')
subprocess.run([
    str(PERCEPTION_PY), '-c',
    'import transformers, torch; assert transformers.__version__ == "4.51.3"; print("✓ Perception-env sẵn sàng! Transformers:", transformers.__version__, "| CUDA:", torch.cuda.is_available())'
], check=True, env=clean_env)


1. Đang tạo perception-env bằng virtualenv...
created virtual environment CPython3.13.16.final.0-64-x86_64 in 384ms
  creator CPython3Posix(dest=/kaggle/temp/mem-adapter-v1/perception-env, clear=False, no_vcs_ignore=False, venv_redirect=None, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.3
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator,XonshActivator


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


2. Đang cài đặt thư viện vào perception-env...


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 83.0 MB/s  0:00:09
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 66.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 62.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 124.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 45.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 85.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 127.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 153.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 57.3 M

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


✓ Perception-env sẵn sàng! Transformers: 4.51.3 | CUDA: True


CompletedProcess(args=['/kaggle/temp/mem-adapter-v1/perception-env/bin/python', '-c', 'import transformers, torch; assert transformers.__version__ == "4.51.3"; print("✓ Perception-env sẵn sàng! Transformers:", transformers.__version__, "| CUDA:", torch.cuda.is_available())'], returncode=0)

## 1. Xác thực nguồn dữ liệu từ 2 Datasets & Quản lý dung lượng
Notebook liên kết 10 demonstrations base từ Dataset 1 qua symlink (không tốn dung lượng đĩa và không download lại từ HF), đồng thời định vị và trích xuất derived mem demonstrations từ Dataset 2.


In [3]:
import json

# 1. Liên kết 10 demonstrations base từ Dataset 1 trong /kaggle/input
print('1. Đang kiểm tra và liên kết Base LIBERO-10 demonstrations từ Dataset 1...')
run(sys.executable, REPO / 'scripts/download_memory_training_sources.py',
    '--base-only', '--base-root', BASE_ROOT, '--input-root', INPUT,
    '--report', WORK / 'base-sources.json')
base_report = json.loads((WORK / 'base-sources.json').read_text())
attached_base = sum(1 for s in base_report['sources'] if s['origin'] in ('attached', 'existing'))
downloaded_base = sum(1 for s in base_report['sources'] if s['origin'] == 'download')
print(f'✓ Đã chuẩn bị {len(base_report["sources"])}/10 task base demonstrations (từ input: {attached_base}, tải từ HF: {downloaded_base})!')

# 2. Định vị và giải nén Derived Mem demonstrations từ Dataset 2
print('2. Đang kiểm tra Derived Mem demonstrations từ Dataset 2...')
derived_zips = sorted(INPUT.rglob('libero_mem_derived_v1_sources.zip'))
if derived_zips:
    print(f'-> Tìm thấy derived archive: {derived_zips[0]}')
    MEM_BUNDLE = SCRATCH / 'derived_sources'
    if not (MEM_BUNDLE / 'derived_raw').is_dir():
        print(f'-> Đang giải nén derived sources sang {MEM_BUNDLE}...')
        extract_checked(derived_zips[0], MEM_BUNDLE)
    MEM_ROOT = MEM_BUNDLE / 'derived_raw'
    SPLIT_LOCK = MEM_BUNDLE / 'split.lock.json'
else:
    # Trường hợp người dùng đính kèm thư mục giải nén
    candidates = [p.parent for p in INPUT.rglob('KITCHEN_SCENE1_1_pick_up_the_bowl_and_place_it_back_on_the_plate_demo.hdf5')]
    assert candidates, 'Không tìm thấy derived sources (zip hoặc folder) trong /kaggle/input!'
    MEM_ROOT = candidates[0]
    lock_cands = list(MEM_ROOT.glob('split.lock.json')) + list(MEM_ROOT.parent.glob('split.lock.json'))
    SPLIT_LOCK = lock_cands[0] if lock_cands else (REPO / 'resources/manifests/libero_mem_t1_t3_base10_split_v1.json')

lock_content = json.loads(SPLIT_LOCK.read_text(encoding='utf-8'))
actual_lock = lock_content.get('lock_sha256') or digest(SPLIT_LOCK)
assert actual_lock == EXPECTED_LOCK, f'Split lock mismatch! Nhận: {actual_lock}, Mong đợi: {EXPECTED_LOCK}'
print(f'✓ Đã xác thực Derived Mem split lock ({EXPECTED_LOCK[:12]}...)!')

# 3. Kiểm tra dung lượng đĩa
free_scratch = shutil.disk_usage(SCRATCH).free / 2**30
free_work = shutil.disk_usage(WORK).free / 2**30
print(f'Dung lượng trống: Scratch = {free_scratch:.1f} GiB | Working = {free_work:.1f} GiB')


{
  "status": "planned_not_hash_verified",
  "sources": [
    {
      "repo": "yifengzhu-hf/LIBERO-datasets",
      "revision": "f13aa24a3da8c43c7225569f28c562979fa0e35a",
      "relative": "libero_10/KITCHEN_SCENE3_turn_on_the_stove_and_put_the_moka_pot_on_it_demo.hdf5",
      "sha256": "6b30906a52a5741e98ef447d27e7066d6c0be4a5f7acd7ecaf1cb7468aca4aa9",
      "size_bytes": 1319613988,
      "root": "/kaggle/temp/mem-adapter-v1/resources/demonstrations",
      "destination": "/kaggle/temp/mem-adapter-v1/resources/demonstrations/libero_10/KITCHEN_SCENE3_turn_on_the_stove_and_put_the_moka_pot_on_it_demo.hdf5",
      "origin": "download",
      "local_path": null
    },
    {
      "repo": "yifengzhu-hf/LIBERO-datasets",
      "revision": "f13aa24a3da8c43c7225569f28c562979fa0e35a",
      "relative": "libero_10/KITCHEN_SCENE4_put_the_black_bowl_in_the_bottom_drawer_of_the_cabinet_and_close_it_demo.hdf5",
      "sha256": "703950f48a3c49dfde61be489ade91527f16e1449b4f29a85f2e51153cef3638",
  

## 2. Cài đặt thư viện huấn luyện và tải assets cần thiết

In [ ]:
# Cài đặt các thư viện phục vụ cho phase huấn luyện trên Kaggle kernel
!pip uninstall -y torchao
!pip install -q "lerobot[smolvla,peft,dataset]==0.6.1" "peft>=0.12.0" "accelerate>=0.34.0" "h5py>=3.10.0" "huggingface-hub>=0.25.0"


In [ ]:
# Tải checkpoint gốc và tokenizer đã pin revision
run(TRAIN_PY, REPO / 'scripts/memory_sft_assets.py', 'fetch', ASSETS)
if RUN_DATA_PREP:
    run(PERCEPTION_PY, REPO / 'scripts/download_memory_detector.py',
        '--output', REPO / 'resources/models/grounding-dino-tiny')


## 3. Khôi phục dữ liệu đã tạo & Hoàn thành phase chuẩn bị dữ liệu (Auto-Resume & Parallel)
- **Song song hóa**: Chạy đồng thời T1 trên GPU 0 và T3 trên GPU 1 (giảm thời gian xuống ~4.5h).
- **Auto-Resume**: Nếu có tiến trình dở dang, tự động phục hồi từng episode đã hoàn tất mà không phải chạy lại từ đầu.
- **Bỏ qua khi đã có Shards**: Nếu dataset shards hoàn chỉnh đã tồn tại (ở session này hoặc được gắn từ session trước), tự động bỏ qua toàn bộ bước này trong 0 giây.


In [ ]:
from concurrent.futures import ThreadPoolExecutor

if not RUN_DATA_PREP:
    print('RUN_DATA_PREP = False. Bỏ qua phase chuẩn bị dữ liệu.')
    # Tìm dataset shards đã hoàn thành từ session trước nếu có
    if not (DATA / 'manifest.json').is_file():
        for cand in INPUT.rglob('manifest.json'):
            try:
                m = json.loads(cand.read_text(encoding='utf-8'))
                if m.get('artifact') == 'mixed_base_mem_sft_shards' and m.get('split_lock_sha256') == EXPECTED_LOCK:
                    print(f'-> Tìm thấy dataset shards hoàn chỉnh từ input: {cand.parent}')
                    DATA = cand.parent
                    break
            except Exception:
                continue
else:
    PREP.mkdir(parents=True, exist_ok=True)

    # 1. Tự động khôi phục dữ liệu preparation nếu có từ session trước trong /kaggle/input
    if not (PREP / 'memory_T1/manifest.json').is_file():
        for cand in INPUT.rglob('memory_T1'):
            if cand.is_dir() and (cand / 'manifest.json').is_file():
                parent_prep = cand.parent
                print(f'-> Khôi phục preparation artifacts từ: {parent_prep} -> {PREP}')
                for item in parent_prep.iterdir():
                    dest = PREP / item.name
                    if not dest.exists():
                        if item.is_dir():
                            shutil.copytree(item, dest)
                        else:
                            shutil.copy2(item, dest)
                break

    # 2. Chạy Perception song song trên 2x GPU T4 (T1 trên GPU 0, T3 trên GPU 1)
    protocol = json.loads((REPO / 'configs/benchmarks/libero_mem_t1_t3_protocol.json').read_text())
    tasks = [(task['alias'], task['name']) for task in protocol['tasks']]
    DETECTOR = REPO / 'resources/models/grounding-dino-tiny'

    def prepare_task(item):
        idx, (alias, task_name) = item
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(idx % GPUS), PYTHONNOUSERSITE='1')
        out_dir = PREP / ('memory_' + alias)
        print(f'-> [GPU {idx % GPUS}] Đang xử lý task {alias} ({task_name})...')
        subprocess.run([
            str(PERCEPTION_PY), '-u', str(REPO / 'scripts/prepare_libero_mem_sft.py'),
            '--source-kind', 'derived_raw', '--data-dir', str(MEM_ROOT),
            '--split-lock', str(SPLIT_LOCK), '--task-name', task_name,
            '--tokenizer-path', str(ASSETS / 'backbone'), '--model-dir', str(DETECTOR),
            '--device', 'cuda', '--output-dir', str(out_dir), '--resume',
        ], cwd=REPO, env=env, check=True)
        print(f'✓ [GPU {idx % GPUS}] Hoàn thành perception task {alias}!')

    print('Bắt đầu xử lý perception song song 2 GPU (giảm thời gian xuống ~4-5h)...')
    with ThreadPoolExecutor(max_workers=GPUS) as pool:
        list(pool.map(prepare_task, enumerate(tasks)))

    # 3. Đóng gói mixed dataset shards (10 base + T1/T3 mem)
    if not (DATA / 'manifest.json').is_file():
        print('Đang đóng gói mixed base + mem shards vào working directory...')
        with tempfile.TemporaryDirectory(dir=SCRATCH) as temp_d:
            stage = Path(temp_d) / 'dataset'
            run(PERCEPTION_PY, REPO / 'scripts/prepare_mixed_memory_sft.py',
                '--split-lock', str(SPLIT_LOCK), '--base-root', str(BASE_ROOT), '--mem-root', str(MEM_ROOT),
                '--tokenizer-path', str(ASSETS / 'backbone'), '--output-dir', str(stage),
                '--memory-index', *(PREP / ('memory_' + alias) for alias, _ in tasks))
            shutil.copytree(stage, DATA, dirs_exist_ok=True)

assert (DATA / 'manifest.json').is_file(), 'Chưa tìm thấy dataset shards! Hãy đặt RUN_DATA_PREP = True hoặc gắn prebuilt dataset vào /kaggle/input.'
# Xác thực toàn vẹn dataset
manifest = json.loads((DATA / 'manifest.json').read_text(encoding='utf-8'))
lock = json.loads((DATA / 'split.lock.json').read_text(encoding='utf-8'))
assert lock.get('lock_sha256') == EXPECTED_LOCK, 'Split lock trong dataset không khớp EXPECTED_LOCK!'
run(TRAIN_PY, REPO / 'scripts/validate_mixed_memory_sft.py', '--data-dir', DATA)
shutil.copy2(DATA / 'manifest.json', WORK / 'dataset-manifest.json')
shutil.copy2(DATA / 'split.lock.json', WORK / 'dataset-split.lock.json')
print('✓ Dataset đã sẵn sàng cho training tại:', DATA)
print('Counts:', manifest['counts'])


## 4. Huấn luyện Adapter (Fine-tuning phase với Auto-Resume & Higher Steps)
Khởi chạy huấn luyện phân tán 2x T4 qua Accelerate với batch size hiệu dụng 32 (batch 2 x accumulation 8 x 2 GPUs), điều kiện hóa memory prompt.
Tự động phát hiện checkpoint cao nhất nếu phiên trước bị gián đoạn để tiếp tục huấn luyện liền mạch.


In [ ]:
if not RUN_TRAINING:
    print('RUN_TRAINING = False. Bỏ qua phase huấn luyện (Session hoàn tất ở bước chuẩn bị dữ liệu).')
else:
    # 1. Cấu hình số steps huấn luyện & tần suất lưu checkpoint
    recipe_path = REPO / 'configs/training/mem_adapter_v1.json'
    recipe = json.loads(recipe_path.read_text(encoding='utf-8'))
    recipe['steps'] = TRAIN_STEPS
    recipe['save_every'] = SAVE_EVERY
    recipe_path.write_text(json.dumps(recipe, indent=2) + '\n', encoding='utf-8')
    print(f'-> Cấu hình training: {TRAIN_STEPS} steps, lưu checkpoint mỗi {SAVE_EVERY} steps.')

    # 2. Thư mục output cố định trong /kaggle/working để lưu an toàn khi bị ngắt
    OUTPUT = WORK / 'output'
    OUTPUT.mkdir(parents=True, exist_ok=True)

    # 3. Cơ chế Auto-Resume an toàn: tự động tìm checkpoint cao nhất nếu phiên trước bị ngắt
    resume_ckpt = None
    if AUTO_RESUME_TRAINING:
        ckpt_candidates = sorted(OUTPUT.glob('checkpoint-*')) + sorted(INPUT.rglob('checkpoint-*'))
        valid_ckpts = []
        for c in ckpt_candidates:
            if c.is_dir() and (c / 'progress.json').is_file() and (c / 'model.safetensors').is_file():
                try:
                    step_num = int(c.name.split('-')[-1])
                    valid_ckpts.append((step_num, c))
                except ValueError:
                    continue
        if valid_ckpts:
            valid_ckpts.sort(key=lambda x: x[0])
            latest_step, latest_dir = valid_ckpts[-1]
            if latest_step < TRAIN_STEPS:
                print(f'-> [AUTO-RESUME] Phát hiện checkpoint trước đó: {latest_dir.name} (step {latest_step}/{TRAIN_STEPS}). Tự động tiếp tục!')
                resume_ckpt = latest_dir
            else:
                print(f'-> Checkpoint {latest_dir.name} đã hoàn thành toàn bộ {TRAIN_STEPS} steps!')

    # 4. Khởi chạy huấn luyện phân tán 2x T4 qua Accelerate
    train_args = [
        str(TRAIN_PY), str(REPO / 'scripts/train_mem_adapter.py'),
        '--data-dir', str(DATA),
        '--assets-dir', str(ASSETS),
        '--output-dir', str(OUTPUT),
        '--gpus', str(GPUS),
        '--steps', str(TRAIN_STEPS),
        '--save-every', str(SAVE_EVERY)
    ]
    if resume_ckpt:
        train_args += ['--resume', str(resume_ckpt)]

    print(f'Bắt đầu huấn luyện adapter tại {OUTPUT}...')
    subprocess.run(train_args, cwd=REPO, check=True)

    print('✓ Huấn luyện hoàn tất! Unmerged adapter đã được xuất tại:', OUTPUT / 'pretrained_model')
    saved_bytes = sum(p.stat().st_size for p in OUTPUT.rglob('*') if p.is_file())
    print(f'Working output GB: {saved_bytes / 1e9:.2f} GB')


## 5. Lưu trữ Artifacts và Hướng dẫn Quản lý Phiên (12h Kaggle Session)

### Khuyên dùng: Quy trình 2 Sessions an toàn khi huấn luyện sâu (>= 5,000 steps):
1. **Session 1 (Data Prep ~4.5h)**:
   - Đặt `RUN_DATA_PREP = True` và `RUN_TRAINING = False`.
   - Chạy toàn bộ notebook. Perception song song 2 GPU và đóng gói shards sẽ hoàn tất sau ~4.5 giờ.
   - Bấm **Save Version** (chọn *Save & Run All* hoặc *Quick Save*). Output shards sẽ lưu tại `/kaggle/working/mem-adapter-v1/dataset`.
2. **Session 2 (Fine-tuning ~8h)**:
   - Mở notebook phiên mới.
   - Bấm **+ Add Input** -> **Your Work** -> chọn Notebook Session 1 (để gắn dataset shards đã đóng gói).
   - Đặt `RUN_DATA_PREP = False` và `RUN_TRAINING = True`.
   - Notebook sẽ tự nhận diện shards có sẵn, bỏ qua 100% bước chuẩn bị dữ liệu và huấn luyện ngay 5,000 steps (~8h) hoàn toàn trọn vẹn trong 12h!

### Cơ chế phòng ngừa nếu bị ngắt giữa chừng:
- **Nếu bị ngắt ở bước Data Prep**: Chạy lại notebook, cơ chế `--resume` sẽ tự nhận diện các episode đã ghi nhận trong `PREP` và chạy tiếp các episode còn thiếu.
- **Nếu bị ngắt ở bước Training**: Gắn output của session bị ngắt làm Input cho session mới. `AUTO_RESUME_TRAINING = True` sẽ tự động tìm checkpoint cao nhất (ví dụ `checkpoint-002500`) và nạp lại để huấn luyện tiếp đến `TRAIN_STEPS` mà không mất công train lại từ đầu!
